# 🌲 Food Delivery Delay Prediction - Model Development & Evaluation
**Author:** Aadi Shah | B.Tech CSE (23000255)  
**Model:** Random Forest Regressor (`sklearn.ensemble.RandomForestRegressor`)  
**Evaluation Partition:** 80% Training (160,000 records) / 20% Testing (40,000 records)

This notebook documents the end-to-end machine learning engineering workflow, including strict anti-leakage preprocessing, model training, test set evaluation metrics (MAE, RMSE, R²), residual error analysis, feature importances, and validation against the Slide 9 demonstration case.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_PATH, MODEL_SAVE_PATH, TARGET_COLUMN
from src.data_loader import load_raw_data, validate_and_clean_data
from src.preprocessing import prepare_train_test_data
from src.train import train_model
from src.evaluate import compute_metrics, plot_feature_importance
from src.predict import DeliveryDelayPredictor

plt.style.use('seaborn-v0_8-whitegrid')
print('Modules loaded successfully.')

## 1. Load Data & Execute 80/20 Leakage-Free Partitioning
To prevent data leakage, OneHotEncoder and temporal transformations are strictly fitted only on the 160,000 training records, then applied to the 40,000 unseen test records.

In [ ]:
raw_df = load_raw_data(filepath=RAW_DATA_PATH)
clean_df, summary = validate_and_clean_data(raw_df)

X_train, X_test, y_train, y_test, preprocessor = prepare_train_test_data(
    clean_df, test_size=0.20, random_state=42, save_processed=False
)

print(f'Training shape: {X_train.shape}')
print(f'Testing shape:  {X_test.shape}')
print(f'Total engineered features: {len(preprocessor.encoded_feature_names)}')

## 2. Load or Train the Random Forest Regressor
Hyperparameters:
- `n_estimators`: 100
- `max_depth`: 20
- `min_samples_split`: 5
- `min_samples_leaf`: 2
- `n_jobs`: -1 (multi-core parallelization)

In [ ]:
if MODEL_SAVE_PATH.exists():
    print(f'Loading pre-trained model package from {MODEL_SAVE_PATH}...')
    pkg = joblib.load(MODEL_SAVE_PATH)
    model = pkg['model']
    preprocessor = pkg['preprocessor']
else:
    print('Training model on 160,000 records...')
    model, pkg = train_model(X_train, y_train, preprocessor)

print('Model ready for evaluation.')

## 3. Evaluation on 40,000 Unseen Test Records
Compute standard continuous regression metrics: MAE, RMSE, R², Median Absolute Error.

In [ ]:
y_pred = model.predict(X_test)
metrics = compute_metrics(y_test, y_pred)

metrics_df = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])
metrics_df

## 4. Goodness of Fit & Residual Diagnostics
Analyze error distribution and verify absence of systematic bias.

In [ ]:
residuals = np.array(y_test) - y_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Actual vs Predicted
sample_idx = np.random.choice(len(y_test), 1500, replace=False)
ax1.scatter(np.array(y_test)[sample_idx], y_pred[sample_idx], alpha=0.3, color='#1f77b4', s=20)
ax1.plot([0, 20], [0, 20], 'r--', linewidth=2, label='Ideal 1:1')
ax1.set_title('Actual vs Predicted Delay (Test Set)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Actual Delay (min)')
ax1.set_ylabel('Predicted Delay (min)')
ax1.legend()

# Subplot 2: Residuals Distribution
sns.histplot(residuals, kde=True, bins=50, color='#8b5cf6', ax=ax2)
ax2.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax2.set_title('Residuals Distribution (Mean ≈ 0)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Residual Error (Actual - Predicted, min)')

plt.tight_layout()
plt.show()

## 5. Feature Importances (Mean Decrease in Impurity)
Inspect the most influential features driving delivery delay.

In [ ]:
feat_imp = pd.DataFrame({
    'feature': preprocessor.encoded_feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 6))
top15 = feat_imp.head(15)
sns.barplot(data=top15, x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('Top 15 Feature Importances (Random Forest MDI)', fontsize=13, fontweight='bold')
plt.show()

top15

## 6. Slide 9 Reference Demonstration & Decision Support (Slide 10)
Validate the model against the representative order specified in Slide 9:
- Food: Pizza | Distance: 10.5 km | Order Value: ₹680
- Vehicle: Bike | Traffic: High | Weather: Rainy | Route Efficiency: 0.45

In [ ]:
predictor = DeliveryDelayPredictor()
example_order = {
    'food_item': 'Pizza',
    'order_time': '24-05-2024',
    'delivery_distance': 10.5,
    'order_value': 680.0,
    'delivery_method': 'Bike',
    'traffic_condition': 'High',
    'weather_condition': 'Rainy',
    'route_taken': 'Route_2',
    'age': 28,
    'gender': 'Male',
    'location': 'Ahmedabad',
    'order_history': 25,
    'customer_rating': 4,
    'preferred_cuisine': 'Italian',
    'order_frequency': 'Weekly',
    'loyalty_program': 'Yes',
    'route_type': 'Mixed',
    'route_efficiency': 0.45,
    'traffic_avoidance': 'No'
}

result = predictor.predict_single(example_order)
print('--- INFERENCE RESULT ---')
print(f"Predicted Delay:     {result['predicted_delay_minutes']} minutes")
print(f"Risk Level:          {result['risk_level']}")
print(f"Actionable Guidance: {result['recommended_action']}")